# TrDPNet a mano · el modelo calculado paso a paso

### Una pasada completa del modelo con números que caben en una hoja

Este notebook reproduce **el mismo cálculo que ejecuta TrDPNet**, pero con una versión en miniatura: 8 puntos en lugar de 8192, 3 tokens geométricos en lugar de 512, dimensión 4 en lugar de 768 y una imagen de 4×4 píxeles en lugar de 224×224. Las operaciones son idénticas; solo cambian las dimensiones.

El propósito es que **cada número sea legible**: toda matriz que aparece puede verificarse con lápiz y papel, y al final del recorrido se comprueba que el resultado coincide, hasta la precisión de la máquina, con el que produce PyTorch ejecutando las mismas operaciones.

| Etapa | Qué se calcula | Dimensiones en miniatura | Dimensiones reales |
|---|---|---|---|
| 1 | Entradas: nube con ruido, imagen y paso de difusión | 8×3, 4×4, escalar | 8192×3, 224×224×3, escalar |
| 2 | Capa SA: de puntos a tokens | 8×3 → 3×4 | 8192×3 → 512×768 |
| 3 | Condición: paso temporal e imagen | 5×4 | 198×768 |
| 4 | Tensores $Q$, $K$ y $V$ | 3×4 y 5×4 | 513×768 y 198×768 |
| 5 | Atención: afinidad, softmax y mezcla | 3×5 | 513×198 |
| 6 | Avance directo y conexión residual | 3×4 | 513×768 |
| 7 | Capa FP: de tokens a puntos | 3×4 → 8×3 | 512×768 → 8192×3 |
| 8 | Guía sin clasificador y paso de muestreo | 8×3 | 8192×3 |

### Convenio de colores

| Color | Significado |
|---|---|
| 🟨 amarillo | pesos de atención, ya normalizados por el softmax |
| 🟥 rojo | matrices de parámetros aprendidos ($W_q$, $W_k$, $W_v$, MLP) |
| 🟦 azul | activaciones: tokens y rasgos intermedios |
| 🟩 verde | entradas y salidas del modelo |
| ⬜ gris | valores auxiliares: distancias, índices, coeficientes |

In [ ]:
# =============================================================================
# 0 · UTILIDADES DE DIBUJO
# =============================================================================
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from IPython.display import HTML, display

np.set_printoptions(precision=2, suppress=True)
torch.set_printoptions(precision=3, sci_mode=False)
plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": False,
                     "figure.facecolor": "white"})

PALETA = {"atencion": "#FFE680", "peso": "#FFFFFF", "activacion": "#DCEBFA",
          "entrada": "#DFF3E0", "aux": "#EFF1F3", "resaltado": "#FFD0C4"}
BORDE = {"atencion": "#C9A227", "peso": "#D0342C", "activacion": "#2F6690",
         "entrada": "#3A7D44", "aux": "#8A94A0", "resaltado": "#E4572E"}

def matriz(ax, A, x, y, celda=0.62, tipo="activacion", titulo=None, fmt="{:.2f}",
           filas=None, cols=None, destacar=None, fs=7.5):
    '''Dibuja una matriz como rejilla de celdas con sus valores, al estilo de un cálculo manual.'''
    A = np.atleast_2d(np.asarray(A, float))
    n, m = A.shape
    for i in range(n):
        for j in range(m):
            resalta = destacar is not None and (i, j) in destacar
            ax.add_patch(Rectangle((x + j * celda, y - (i + 1) * celda), celda, celda,
                                   fc=PALETA["resaltado"] if resalta else PALETA[tipo],
                                   ec=BORDE["resaltado"] if resalta else BORDE[tipo],
                                   lw=1.4 if resalta else 0.8, zorder=2))
            ax.text(x + (j + .5) * celda, y - (i + .5) * celda, fmt.format(A[i, j]),
                    ha="center", va="center", fontsize=fs, zorder=3)
    if titulo:
        ax.text(x + m * celda / 2, y + .18, titulo, ha="center", va="bottom", fontsize=fs + 1.5,
                weight="bold", color=BORDE[tipo])
    if filas:
        for i, t in enumerate(filas):
            ax.text(x - .12, y - (i + .5) * celda, t, ha="right", va="center", fontsize=fs - .5,
                    color="#40566b")
    if cols:
        for j, t in enumerate(cols):
            ax.text(x + (j + .5) * celda, y - n * celda - .12, t, ha="center", va="top",
                    fontsize=fs - .5, color="#40566b")
    return x + m * celda, y - n * celda          # esquina inferior derecha

def flecha(ax, p, q, texto=None, color="#16324F", fs=8, desvio=0.0):
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=10, lw=1.2,
                                 color=color, zorder=4,
                                 connectionstyle=f"arc3,rad={desvio}"))
    if texto:
        ax.text((p[0] + q[0]) / 2, (p[1] + q[1]) / 2 + .18, texto, ha="center", va="bottom",
                fontsize=fs, color=color)

def lienzo(ancho=14, alto=5, xlim=(0, 28), ylim=(-10, 2)):
    fig, ax = plt.subplots(figsize=(ancho, alto))
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.axis("off"); ax.set_aspect("equal")
    return fig, ax

def tarjeta(titulo, muestra, analisis, interpretacion, sintesis=None):
    '''Presenta un resultado en tres bloques: qué muestra, análisis e interpretación.'''
    li = lambda xs: "".join(f"<li style='margin:3px 0'>{x}</li>" for x in xs)
    pie = (f"<div style='padding:8px 14px;background:#1f3b57;color:#fff'>📌 <b>Síntesis:</b> "
           f"{sintesis}</div>") if sintesis else ""
    display(HTML(f'''
<div style="border:1px solid #c9d3dd;border-radius:10px;margin:10px 0;overflow:hidden;
            font-family:Segoe UI,Arial,sans-serif;font-size:13px;line-height:1.4">
  <div style="background:#16324f;color:#fff;padding:8px 14px;font-weight:600;font-size:14px">{titulo}</div>
  <div style="display:grid;grid-template-columns:1fr 1.2fr 1.2fr">
    <div style="padding:10px 14px;background:#eaf2fb"><b>📊 Qué muestra</b><p style="margin:6px 0">{muestra}</p></div>
    <div style="padding:10px 14px;background:#fff7e3"><b>🔎 Análisis</b><ul style="padding-left:18px;margin:6px 0">{li(analisis)}</ul></div>
    <div style="padding:10px 14px;background:#e9f6ec"><b>💡 Interpretación</b><ul style="padding-left:18px;margin:6px 0">{li(interpretacion)}</ul></div>
  </div>{pie}
</div>'''))

print("Utilidades de dibujo listas.")

# 1 · Las entradas

El modelo recibe tres cosas en cada paso de la inferencia:

1. **Una nube con ruido** $x_t \in \mathbb{R}^{N\times 3}$. Aquí $N=8$ puntos, con coordenadas elegidas para que formen una silueta reconocible: una base plana y un respaldo, es decir, una silla vista en miniatura.
2. **Una imagen** de la que se extraen tokens. Aquí es una rejilla de 4×4 píxeles que se divide en **4 parches de 2×2**, igual que el ViT divide 224×224 en parches de 16×16.
3. **El paso de difusión** $t$, un entero que indica cuánto ruido queda. Aquí $t=3$ sobre un total de $T=6$.

Las coordenadas se muestran con dos decimales; todas las operaciones posteriores parten exactamente de estos valores.

In [ ]:
# =============================================================================
# 1 · ENTRADAS DEL MODELO
# =============================================================================
torch.manual_seed(7)
N, M_TOK, D, H, I_PAR, T_TOT, T_PASO = 8, 3, 4, 2, 4, 6, 3
DH = D // H

# --- nube limpia: una "silla" de ocho puntos ----------------------------------
x0 = np.array([[-1.0, 0.0, -1.0], [1.0, 0.0, -1.0], [-1.0, 0.0, 1.0], [1.0, 0.0, 1.0],
               [-1.0, 1.5, -1.0], [1.0, 1.5, -1.0], [-1.0, 0.8, -1.0], [1.0, 0.8, -1.0]]) * 0.5

# --- programa de ruido en miniatura -------------------------------------------
betas = np.linspace(0.10, 0.50, T_TOT)
alphas = 1 - betas
ab = np.cumprod(alphas)
ruido = np.round(np.random.default_rng(3).normal(0, 1, (N, 3)), 2)
xt = np.round(np.sqrt(ab[T_PASO]) * x0 + np.sqrt(1 - ab[T_PASO]) * ruido, 2)

# --- imagen 4x4 y sus cuatro parches ------------------------------------------
imagen = np.array([[0, 0, 1, 1],
                   [0, 1, 1, 0],
                   [1, 1, 0, 1],
                   [0, 1, 0, 1]], float)
parches = np.stack([imagen[:2, :2].ravel(), imagen[:2, 2:].ravel(),
                    imagen[2:, :2].ravel(), imagen[2:, 2:].ravel()])       # (4 parches, 4 valores)

fig, ax = lienzo(15, 4.6, (0, 30), (-6.6, 2.4))
matriz(ax, x0, 0.6, 0, titulo="$x_0$ · objeto limpio (8×3)", tipo="entrada",
       cols=["x", "y", "z"], filas=[f"p{i+1}" for i in range(N)])
flecha(ax, (3.4, -2.5), (5.2, -2.5), "+ ruido")
matriz(ax, xt, 5.6, 0, titulo=f"$x_t$ · entrada real (t={T_PASO})", tipo="entrada",
       cols=["x", "y", "z"])
matriz(ax, imagen, 10.5, 0, titulo="imagen 4×4", tipo="entrada", fmt="{:.0f}", celda=0.7)
ax.add_patch(Rectangle((10.5, -1.4), 1.4, 1.4, fill=False, ec="#E4572E", lw=2, zorder=5))
ax.text(11.2, 0.55, "parche 1", ha="center", fontsize=7, color="#E4572E")
matriz(ax, parches, 14.4, 0, titulo="4 parches aplanados (4×4)", tipo="activacion", fmt="{:.0f}",
       filas=[f"parche {i+1}" for i in range(4)])
matriz(ax, np.array([[T_PASO]]), 19.8, 0, titulo="paso $t$", tipo="aux", fmt="{:.0f}", celda=0.8)
matriz(ax, np.array([np.round(ab, 3)]), 21.6, 0, titulo=r"$\bar{\alpha}_t$ (programa de ruido)",
       tipo="aux", fmt="{:.2f}", celda=0.72, cols=[f"t={i}" for i in range(T_TOT)],
       destacar={(0, T_PASO)})
plt.tight_layout(); plt.show()

print("Coeficientes en t =", T_PASO, ":  raíz(ab) =", round(float(np.sqrt(ab[T_PASO])), 3),
      " · raíz(1-ab) =", round(float(np.sqrt(1 - ab[T_PASO])), 3))

tarjeta("1 · Las tres entradas",
        "Objeto limpio, nube contaminada con ruido en el paso t, imagen dividida en parches y coeficientes "
        "del programa de difusión.",
        [f"El modelo recibe <b>$x_t$</b>, no $x_0$: el objeto limpio aparece aquí solo como referencia.",
         f"Con t = {T_PASO}, la señal pesa {float(np.sqrt(ab[T_PASO])):.2f} y el ruido "
         f"{float(np.sqrt(1-ab[T_PASO])):.2f}.",
         "La imagen se trocea en parches sin solape; cada parche se aplana en un vector."],
        ["Los dos coeficientes cumplen que sus cuadrados suman uno, de modo que la escala de la nube se "
         "mantiene mientras cambia la proporción entre señal y ruido.",
         "Trocear la imagen en parches es lo que permite tratarla como una secuencia y, por tanto, "
         "atenderla con el mismo mecanismo que a los puntos.",
         "El paso t entra como un dato más: sin él, el modelo no sabría cuánto ruido debe retirar."],
        "Tres entradas de naturaleza distinta que deben acabar representadas en un mismo espacio vectorial.")

# 2 · Capa SA: de 8 puntos a 3 tokens

La capa de abstracción convierte la nube en una secuencia corta de vectores. Opera en cuatro pasos, todos visibles en los números de abajo:

1. **Muestreo por punto más lejano.** Se elige un punto inicial y, a continuación, aquel que está más lejos de los ya elegidos. El procedimiento reparte los centros por toda la nube en lugar de concentrarlos donde hay más densidad.
2. **Agrupación por esfera.** Alrededor de cada centro se recogen los $K$ vecinos más próximos dentro de un radio. Aquí $K=3$.
3. **Coordenadas relativas.** A cada vecino se le resta su centro: $p_k - c_j$. El token describe así la *forma local*, no la posición absoluta.
4. **MLP compartido y max-pool.** Cada vecino pasa por la misma matriz $W_{SA}$ y, de los $K$ resultados, se conserva el máximo componente a componente:

$$z_j = \max_{k=1..K}\ \left[(p_k - c_j)\,W_{SA}\right]$$

El *max-pool* es lo que hace el resultado **invariante al orden** de los vecinos: reordenarlos no cambia el máximo. Esa propiedad es la que permite tratar una nube, que carece de orden natural, como una secuencia.

In [ ]:
# =============================================================================
# 2 · CAPA SA: MUESTREO, AGRUPACIÓN Y MAX-POOL
# =============================================================================
def fps(P, m):
    '''Muestreo por punto más lejano: devuelve los índices de los m centros.'''
    idx = [0]
    d = np.linalg.norm(P - P[0], axis=1)
    for _ in range(m - 1):
        j = int(np.argmax(d)); idx.append(j)
        d = np.minimum(d, np.linalg.norm(P - P[j], axis=1))
    return idx

def vecinos(P, centros, k):
    '''Los k vecinos más próximos de cada centro.'''
    Dm = np.linalg.norm(P[None, :, :] - centros[:, None, :], axis=-1)
    return np.argsort(Dm, axis=1)[:, :k], Dm

idx_c = fps(xt, M_TOK)
centros = xt[idx_c]
K = 3
idx_v, Dist = vecinos(xt, centros, K)

W_SA = np.array([[1., 0., -1., 0.],
                 [0., 1., 0., 1.],
                 [1., 1., 0., -1.]])                  # (3 coordenadas → 4 dimensiones)

rel = np.stack([xt[idx_v[j]] - centros[j] for j in range(M_TOK)])      # (M, K, 3)
proy = np.round(rel @ W_SA, 2)                                        # (M, K, 4)
z = np.round(proy.max(axis=1), 2)                                     # (M, 4) max-pool

fig, ax = lienzo(15.5, 5.6, (0, 31), (-8.4, 2.6))
matriz(ax, xt, 0.6, 0, titulo="$x_t$ (8×3)", tipo="entrada", cols=["x", "y", "z"],
       filas=[f"p{i+1}" for i in range(N)], destacar={(i, j) for i in idx_c for j in range(3)})
ax.text(1.8, -6.0, f"centros elegidos: p{idx_c[0]+1}, p{idx_c[1]+1}, p{idx_c[2]+1}",
        fontsize=7.5, color="#E4572E")
flecha(ax, (3.2, -2.6), (5.0, -2.6), "agrupar")

y0 = 0
for j in range(M_TOK):
    matriz(ax, rel[j], 5.4, y0, titulo=f"$p_k - c_{j+1}$  (3×3)" if j == 0 else None,
           tipo="aux", celda=0.55, filas=[f"v{k+1}" for k in range(K)], fs=7)
    matriz(ax, proy[j], 8.4, y0, titulo="$(p_k-c_j)\\,W_{SA}$  (3×4)" if j == 0 else None,
           tipo="activacion", celda=0.55, fs=7)
    matriz(ax, z[j:j + 1], 12.4, y0, titulo="max-pool → token" if j == 0 else None,
           tipo="activacion", celda=0.55, fs=7,
           destacar={(0, c) for c in range(D)})
    ax.text(12.2, y0 - 0.28, f"$z_{j+1}$", ha="right", fontsize=8, color="#2F6690")
    y0 -= 2.3
matriz(ax, W_SA, 16.4, 0, titulo="$W_{SA}$ (3×4)", tipo="peso", celda=0.62,
       filas=["x", "y", "z"])
flecha(ax, (11.9, -1.0), (12.3, -1.0))
matriz(ax, z, 20.5, 0, titulo="tokens $z$ (3×4)", tipo="activacion",
       filas=[f"$z_{j+1}$" for j in range(M_TOK)])
plt.tight_layout(); plt.show()

# --- comprobación de la invarianza al orden ------------------------------------
perm = np.random.default_rng(0).permutation(K)
z_perm = np.round((rel[:, perm, :] @ W_SA).max(axis=1), 2)
print("¿El token cambia si se reordenan los vecinos?",
      "no" if np.allclose(z, z_perm) else "sí", " · diferencia máxima:",
      float(np.abs(z - z_perm).max()))

tarjeta("2 · De 8 puntos a 3 tokens",
        "Centros elegidos, coordenadas relativas de sus vecinos, proyección por el MLP compartido y "
        "máximo componente a componente.",
        [f"Los {N} puntos quedan resumidos en {M_TOK} tokens de {D} dimensiones.",
         "Las coordenadas se toman siempre relativas al centro: el token describe forma local, no posición.",
         "Reordenar los vecinos no altera el resultado, como confirma la comprobación numérica."],
        ["El max-pool es la pieza que aporta invarianza a permutaciones, requisito imprescindible al "
         "trabajar con nubes de puntos, que carecen de orden.",
         "La reducción de puntos a tokens es también lo que hace viable la atención: su coste crece con el "
         "cuadrado de la longitud de la secuencia.",
         "En el modelo real la proporción es la misma operación a otra escala: 8192 puntos y 512 centros."],
        "La geometría se convierte en una secuencia corta de vectores sin perder la forma local.")

# 3 · La condición: paso temporal e imagen en un mismo espacio

La condición $c$ es la secuencia que el modelo consultará. Se construye en dos partes:

**El paso temporal** se codifica con senos y cosenos de frecuencias decrecientes, la misma codificación posicional del Transformer original:

$$e_{2i}=\sin\!\left(\frac{t}{10000^{2i/d}}\right),\qquad e_{2i+1}=\cos\!\left(\frac{t}{10000^{2i/d}}\right)$$

Frecuencias distintas producen vectores distinguibles para cada $t$, sin necesidad de aprender una tabla.

**Los parches de imagen** se proyectan al mismo espacio con una matriz $W_{img}$, que en el modelo real es la convolución de parches del ViT seguida de sus doce bloques.

El resultado se apila: un token temporal más cuatro de imagen, cinco en total. Todos comparten la dimensión de los tokens geométricos, y esa coincidencia es la que permite atender unos a otros.

In [ ]:
# =============================================================================
# 3 · CONSTRUCCIÓN DE LA CONDICIÓN
# =============================================================================
def codifica_t(t, d):
    '''Codificación sinusoidal del paso de difusión.'''
    i = np.arange(d // 2)
    f = 1.0 / (10000 ** (2 * i / d))
    return np.round(np.concatenate([np.sin(t * f), np.cos(t * f)]), 2)

t_tok = codifica_t(T_PASO, D)
W_img = np.array([[1., 0., 1., 0.],
                  [0., 1., 0., 1.],
                  [1., 1., 0., 0.],
                  [0., 0., 1., 1.]])                  # (4 valores del parche → 4 dimensiones)
img_tok = np.round(parches @ W_img, 2)
c = np.round(np.vstack([t_tok, img_tok]), 2)          # (5, 4) condición completa

fig, ax = lienzo(15.5, 4.4, (0, 31), (-6.4, 2.4))
matriz(ax, np.array([[T_PASO]]), 0.6, 0, titulo="paso $t$", tipo="aux", fmt="{:.0f}", celda=0.8)
flecha(ax, (1.6, -0.4), (3.2, -0.4), "sen/cos")
matriz(ax, t_tok[None], 3.4, 0, titulo="token temporal (1×4)", tipo="activacion")
matriz(ax, parches, 3.4, -2.2, titulo="parches (4×4)", tipo="entrada", fmt="{:.0f}")
matriz(ax, W_img, 8.2, -2.2, titulo="$W_{img}$ (4×4)", tipo="peso")
flecha(ax, (7.9, -4.0), (8.1, -4.0))
flecha(ax, (11.2, -4.0), (12.6, -4.0), "=")
matriz(ax, img_tok, 12.8, -2.2, titulo="tokens de imagen (4×4)", tipo="activacion")
flecha(ax, (16.2, -1.2), (18.2, -2.2), desvio=-0.2)
flecha(ax, (6.2, -1.0), (18.2, -1.0), desvio=0.12)
matriz(ax, c, 18.4, 0, titulo="condición $c$ (5×4)", tipo="activacion",
       filas=["$t$", "img 1", "img 2", "img 3", "img 4"])
matriz(ax, z, 23.6, 0, titulo="tokens de nube $x$ (3×4)", tipo="activacion",
       filas=[f"$z_{j+1}$" for j in range(M_TOK)])
ax.text(21.0, -4.6, "misma dimensión: $d = %d$" % D, ha="center", fontsize=9, color="#16324F")
plt.tight_layout(); plt.show()

tarjeta("3 · Todo en el mismo espacio",
        "Codificación del paso de difusión, proyección de los parches y secuencia de condición resultante, "
        "junto a los tokens geométricos.",
        [f"La condición tiene {c.shape[0]} tokens: uno temporal y {I_PAR} de imagen.",
         "Los valores del token temporal proceden de senos y cosenos de frecuencias distintas.",
         f"Condición y tokens de nube comparten dimensión {D}."],
        ["Compartir dimensión no es un detalle de implementación: es el requisito para que un token pueda "
         "consultar a otro mediante un producto escalar.",
         "La codificación sinusoidal permite representar cualquier paso sin parámetros adicionales y "
         "mantiene distinguibles valores próximos de t.",
         "En el modelo real la proyección de parches la realiza un ViT congelado de doce bloques; la "
         "operación conceptual es la misma."],
        "La imagen y el paso de difusión se convierten en tokens del mismo tipo que la geometría.")

# 4 · Los tensores $Q$, $K$ y $V$

Tres proyecciones lineales de las secuencias, con la asimetría que define la atención cruzada:

$$Q = x\,W_q \quad \text{(de los tokens de nube)}, \qquad K = c\,W_k,\quad V = c\,W_v \quad \text{(de la condición)}$$

Cada fila de $Q$ es **una pregunta**: qué necesita ese token. Cada fila de $K$ es **una etiqueta**: qué ofrece ese elemento de la condición. Cada fila de $V$ es **el contenido** que se entrega si la etiqueta encaja con la pregunta.

Antes de proyectar, cada fila se **normaliza**: se le resta su media y se divide por su desviación típica. Es la capa de normalización que precede a cada subbloque en el modelo real, y su efecto aquí es visible: mantiene las afinidades en un rango donde el softmax discrimina en lugar de saturarse.

Las tres matrices $W$ contienen los parámetros aprendidos. En el modelo real están concatenadas en una sola matriz de $768\times2304$ por eficiencia, pero matemáticamente son tres proyecciones independientes, como se muestra aquí.

La división en cabezas parte cada fila en $H$ trozos que atienden por separado. Con $d=4$ y $H=2$, cada cabeza trabaja en $d_h=2$ dimensiones.

In [ ]:
# =============================================================================
# 4 · GENERACIÓN DE Q, K Y V
# =============================================================================
def norma(A, eps=1e-5):
    '''Normalización por filas: media cero y desviación unidad (capa LayerNorm).'''
    A = np.atleast_2d(A)
    return np.round((A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + eps), 2)

zn, cn = norma(z), norma(c)          # entradas normalizadas de los subbloques
Wq = np.array([[1., 0., 0., 1.], [0., 1., 1., 0.], [1., 0., 1., 0.], [0., 1., 0., 1.]])
Wk = np.array([[2., 0., 1., 0.], [0., 1., 0., 1.], [1., 0., 0., 1.], [0., 1., 1., 0.]])
Wv = np.array([[1., 0., 0., 0.], [0., 2., 0., 0.], [0., 0., 1., 0.], [0., 0., 0., 2.]])

Q = np.round(zn @ Wq, 2)         # (3, 4) · preguntas, desde los tokens de nube normalizados
Kk = np.round(cn @ Wk, 2)        # (5, 4) · etiquetas, desde la condición normalizada
V = np.round(cn @ Wv, 2)         # (5, 4) · contenidos, desde la condición normalizada

fig, ax = lienzo(15.5, 5.2, (0, 31), (-8.6, 2.6))
matriz(ax, zn, 0.6, 0, titulo="$x$ normalizado (3×4)", tipo="activacion")
matriz(ax, Wq, 4.2, 0, titulo="$W_q$", tipo="peso")
flecha(ax, (7.0, -1.3), (8.4, -1.3), "=")
matriz(ax, Q, 8.6, 0, titulo="$Q$ (3×4)", tipo="activacion",
       filas=[f"$q_{j+1}$" for j in range(M_TOK)])
ax.text(10.0, -2.6, "preguntas", fontsize=8, color="#2F6690", ha="center")

matriz(ax, cn, 0.6, -4.4, titulo="$c$ normalizada (5×4)", tipo="activacion",
       filas=["$t$", "i1", "i2", "i3", "i4"])
matriz(ax, Wk, 4.2, -4.4, titulo="$W_k$", tipo="peso")
flecha(ax, (7.0, -6.0), (8.4, -6.0), "=")
matriz(ax, Kk, 8.6, -4.4, titulo="$K$ (5×4)", tipo="activacion")
ax.text(10.0, -8.0, "etiquetas", fontsize=8, color="#2F6690", ha="center")

matriz(ax, Wv, 13.8, -4.4, titulo="$W_v$", tipo="peso")
flecha(ax, (16.6, -6.0), (18.0, -6.0), "=")
matriz(ax, V, 18.2, -4.4, titulo="$V$ (5×4)", tipo="activacion")
ax.text(19.6, -8.0, "contenidos", fontsize=8, color="#2F6690", ha="center")

ax.add_patch(FancyBboxPatch((23.2, -3.4), 7.0, 5.4, boxstyle="round,pad=0.25",
                            fc="#FFF7E3", ec="#C9A227", lw=1))
ax.text(26.7, 1.5, "La asimetría, en una línea", ha="center", fontsize=9, weight="bold")
ax.text(23.5, 0.6, "$Q \\leftarrow$ tokens de la nube", fontsize=8.5)
ax.text(23.5, -0.3, "$K,\\,V \\leftarrow$ condición (imagen + $t$)", fontsize=8.5)
ax.text(23.5, -1.4, "La nube pregunta;", fontsize=8.5)
ax.text(23.5, -2.2, "la imagen responde.", fontsize=8.5)
plt.tight_layout(); plt.show()

tarjeta("4 · Las tres proyecciones",
        "Producto de cada secuencia por su matriz de parámetros: las preguntas salen de la nube; las "
        "etiquetas y los contenidos, de la condición.",
        [f"$Q$ tiene {Q.shape[0]} filas, una por token de nube; $K$ y $V$ tienen {Kk.shape[0]}, una por "
         "elemento de la condición.",
         "Las tres matrices de parámetros son independientes: definen qué significa «preguntar», "
         "«etiquetar» y «contener» en esta capa.",
         "En auto-atención las tres provendrían de la misma secuencia; aquí no, y ahí está la diferencia."],
        ["Esa asimetría es el mecanismo exacto por el que la información visual entra en la geometría: "
         "sin ella, la imagen no tendría vía de acceso al modelo.",
         "Las matrices de parámetros son lo que el entrenamiento ajusta; cambiarlas cambia qué se "
         "considera afín a qué.",
         "El número de filas de Q y de K no tiene por qué coincidir: la atención relaciona secuencias de "
         "longitudes distintas."],
        "La nube formula preguntas y la imagen aporta etiquetas y contenidos: eso es la atención cruzada.")

# 5 · La atención, celda a celda

Tres operaciones encadenadas:

$$S = \frac{QK^{\top}}{\sqrt{d_h}} \qquad\longrightarrow\qquad A = \mathrm{softmax}(S) \qquad\longrightarrow\qquad O = A\,V$$

**La matriz de afinidad $S$.** Cada celda $S_{ij}$ es el producto escalar entre la pregunta $i$ y la etiqueta $j$: cuanto mayor, más encajan. La división por $\sqrt{d_h}$ evita que el valor crezca con la dimensión: si las componentes tienen varianza unidad, el producto escalar tiene varianza $d_h$, y sin corregirlo el softmax se saturaría.

**La matriz de pesos $A$.** El softmax se aplica **por filas**: cada token de la nube reparte una unidad de atención entre los cinco elementos de la condición. Las filas suman exactamente uno, y eso se comprueba en la salida numérica.

**La salida $O$.** Cada fila es la media ponderada de los contenidos, con los pesos de esa fila. No se selecciona un elemento: se mezclan todos según cuánto encajen.

In [ ]:
# =============================================================================
# 5 · AFINIDAD, SOFTMAX Y MEZCLA
# =============================================================================
S = np.round(Q @ Kk.T / np.sqrt(DH), 2)                       # (3, 5) afinidades
A = np.round(np.exp(S - S.max(1, keepdims=True)) /
             np.exp(S - S.max(1, keepdims=True)).sum(1, keepdims=True), 2)
O = np.round(A @ V, 2)                                        # (3, 4) salida de la atención

fig, ax = lienzo(15.5, 3.4, (0, 31), (-3.8, 2.6))
matriz(ax, Q, 0.6, 0, titulo="$Q$ (3×4)", tipo="activacion")
ax.text(3.4, -1.4, "×", fontsize=13, ha="center")
matriz(ax, Kk.T, 4.0, 0, titulo="$K^{\\top}$ (4×5)", tipo="activacion")
flecha(ax, (7.6, -1.4), (9.0, -1.4), "$/\\sqrt{d_h}$")
matriz(ax, S, 9.2, 0, titulo="$S$ · afinidad (3×5)", tipo="aux",
       cols=["$t$", "i1", "i2", "i3", "i4"], destacar={(0, int(np.argmax(S[0])))})
flecha(ax, (12.8, -1.4), (14.4, -1.4), "softmax\npor filas")
matriz(ax, A, 14.6, 0, titulo="$A$ · pesos (3×5)", tipo="atencion",
       cols=["$t$", "i1", "i2", "i3", "i4"], filas=[f"$z_{j+1}$" for j in range(M_TOK)])
ax.text(16.2, -2.6, f"cada fila suma {A.sum(1)[0]:.2f}", fontsize=7.5, ha="center", color="#C9A227")
ax.text(18.4, -1.4, "×", fontsize=13, ha="center")
matriz(ax, V, 19.0, 0, titulo="$V$ (5×4)", tipo="activacion")
flecha(ax, (22.6, -1.4), (24.0, -1.4), "=")
matriz(ax, O, 24.2, 0, titulo="$O$ · salida (3×4)", tipo="activacion")
plt.tight_layout(); plt.show()

# --- el cálculo de una sola celda, desarrollado --------------------------------
i, j = 0, int(np.argmax(A[0]))
det = " + ".join(f"{Q[i,k]:.2f}·{Kk[j,k]:.2f}" for k in range(D))
print(f"Celda S[{i+1},{j+1}] = ({det}) / raíz({DH}) = {S[i,j]:.2f}")
print(f"Tras el softmax de la fila {i+1}: A[{i+1},{j+1}] = {A[i,j]:.2f}  "
      f"(el mayor peso de esa fila)")
print("Suma de cada fila de A:", np.round(A.sum(1), 3))
print("\nFila 1 de la salida = mezcla de los contenidos:")
for j2 in range(A.shape[1]):
    print(f"   {A[0,j2]:.2f} × [{' '.join(f'{v:5.2f}' for v in V[j2])}]")
print("   " + "-" * 34 + f"\n   = [{' '.join(f'{v:5.2f}' for v in O[0])}]")

tarjeta("5 · Cada token de la nube consulta la imagen",
        "Matriz de afinidad, pesos tras el softmax y mezcla ponderada de los contenidos.",
        [f"La afinidad más alta de la primera fila corresponde al elemento {j+1} de la condición.",
         "Las filas de $A$ suman uno: cada token reparte una unidad de atención entre los cinco elementos.",
         "La salida de cada token es una combinación de los cinco contenidos, no una selección."],
        ["Un reparto poco concentrado no significa que la atención sea inútil: la salida sigue siendo una "
         "mezcla informativa, y así es como el modelo resume la imagen completa.",
         "La normalización por filas garantiza que la escala de la salida no dependa del número de "
         "elementos de la condición.",
         "En el modelo real esta matriz tiene 513 filas y 198 columnas, pero el cálculo es exactamente "
         "este."],
        "La atención es una media ponderada cuyos pesos decide el propio dato mediante productos escalares.")

# 6 · Conexión residual y avance directo

La salida de la atención no sustituye a los tokens: se **suma** a ellos.

$$x' = x + \mathrm{Atención}(x, c), \qquad x'' = x' + \mathrm{MLP}(x')$$

La suma es la conexión residual. Su efecto es que cada subbloque aporta una **corrección** sobre lo que ya había, en lugar de reemplazarlo. Si la corrección fuese nula, el token pasaría intacto, y eso permite apilar bloques sin degradar la señal.

El avance directo aplica la misma transformación a cada token por separado, sin mirar a los demás: expande la dimensión, aplica una no linealidad y vuelve a comprimir. Es donde el modelo transforma cada token de forma individual, después de que la atención haya puesto en común la información.

In [ ]:
# =============================================================================
# 6 · RESIDUAL Y AVANCE DIRECTO
# =============================================================================
W1 = np.array([[1., 0., -1., 1., 0., 1., -1., 0.],
               [0., 1., 1., 0., -1., 0., 1., 1.],
               [1., -1., 0., 1., 1., 0., 0., -1.],
               [0., 1., 1., -1., 0., 1., 1., 0.]])          # 4 → 8
W2 = np.round(W1.T * 0.25, 2)                                # 8 → 4

x1 = np.round(z + O, 2)                                      # residual tras la atención
oculto = np.round(np.maximum(x1 @ W1, 0), 2)                 # avance directo con ReLU
mlp = np.round(oculto @ W2, 2)
x2 = np.round(x1 + mlp, 2)                                   # residual tras el avance directo

fig, ax = lienzo(15.5, 3.4, (0, 31), (-3.8, 2.6))
matriz(ax, z, 0.6, 0, titulo="$x$ (3×4)", tipo="activacion")
ax.text(3.3, -1.3, "+", fontsize=14, ha="center")
matriz(ax, O, 3.8, 0, titulo="atención", tipo="atencion")
flecha(ax, (6.6, -1.3), (7.8, -1.3), "=")
matriz(ax, x1, 8.0, 0, titulo="$x'$ · residual", tipo="activacion")
flecha(ax, (10.8, -1.3), (12.0, -1.3), "$W_1$, ReLU")
matriz(ax, oculto, 12.2, 0, titulo="oculto (3×8)", tipo="activacion", celda=0.52, fs=6.5)
flecha(ax, (16.6, -1.3), (17.8, -1.3), "$W_2$")
matriz(ax, mlp, 18.0, 0, titulo="MLP($x'$)", tipo="activacion")
ax.text(20.8, -1.3, "+", fontsize=14, ha="center")
matriz(ax, x1, 21.4, 0, titulo="$x'$", tipo="activacion")
flecha(ax, (24.2, -1.3), (25.4, -1.3), "=")
matriz(ax, x2, 25.6, 0, titulo="$x''$ · salida del bloque", tipo="activacion")
plt.tight_layout(); plt.show()

nulos = int((oculto == 0).sum())
print(f"Valores anulados por la ReLU: {nulos} de {oculto.size} "
      f"({100*nulos/oculto.size:.0f} %)")
print("Norma media de los tokens · antes:", round(float(np.linalg.norm(z, axis=1).mean()), 2),
      "· tras la atención:", round(float(np.linalg.norm(x1, axis=1).mean()), 2),
      "· tras el bloque completo:", round(float(np.linalg.norm(x2, axis=1).mean()), 2))

tarjeta("6 · El bloque completo, sumando en lugar de sustituir",
        "Suma residual de la atención, expansión con no linealidad, compresión y segunda suma residual.",
        [f"La ReLU anula el {100*nulos/oculto.size:.0f} % de los valores intermedios.",
         "La norma de los tokens crece de forma moderada al atravesar el bloque.",
         "Cada subbloque aporta un término que se suma, nunca reemplaza al anterior."],
        ["La conexión residual permite apilar bloques: si un subbloque no tiene nada que aportar, su "
         "contribución tiende a cero y la señal pasa intacta.",
         "La atención pone en común información entre tokens; el avance directo transforma cada token por "
         "separado. Esa alternancia entre mezcla y transformación individual es la estructura del "
         "Transformer.",
         "El modelo real repite esta estructura cuatro veces, y la condición vuelve a consultarse en cada "
         "repetición."],
        "Un bloque Transformer corrige los tokens dos veces: primero con lo que aportan los demás, después con lo propio.")

# 7 · Capa FP: de 3 tokens a 8 vectores de ruido

El Transformer trabaja con tres tokens, pero la salida debe tener un valor por cada uno de los ocho puntos. La capa de propagación deshace la reducción interpolando:

$$f(p)=\sum_{i=1}^{3} w_i\,f(c_i), \qquad w_i=\frac{1/d_i}{\sum_j 1/d_j}$$

Cada punto recibe una combinación de los tokens, ponderada por el **inverso de su distancia** a cada centro: cuanto más cerca está de un centro, más pesa ese token. Los pesos se normalizan para que sumen uno, de modo que una señal constante se recupera intacta, y esa propiedad se comprueba numéricamente.

Después, un MLP reduce cada vector interpolado a tres canales: el vector de ruido estimado para ese punto.

In [ ]:
# =============================================================================
# 7 · INTERPOLACIÓN Y SALIDA
# =============================================================================
Dpc = np.linalg.norm(xt[:, None, :] - centros[None, :, :], axis=-1)      # (8, 3) distancias
Wint = 1.0 / np.maximum(Dpc, 1e-6)
Wint = np.round(Wint / Wint.sum(1, keepdims=True), 2)                    # pesos normalizados
interp = np.round(Wint @ x2, 2)                                          # (8, 4)
W_out = np.array([[1., 0., 0.], [0., 1., 0.], [0., 0., 1.], [0.5, -0.5, 0.5]])
eps_cond = np.round(interp @ W_out, 2)                                   # (8, 3) ruido estimado

fig, ax = lienzo(15.5, 5.2, (0, 31), (-8.6, 2.6))
matriz(ax, np.round(Dpc, 2), 0.6, 0, titulo="distancias punto→centro (8×3)", tipo="aux",
       filas=[f"p{i+1}" for i in range(N)], cols=["$c_1$", "$c_2$", "$c_3$"])
flecha(ax, (3.2, -2.8), (4.6, -2.8), "$1/d$\nnormalizado")
matriz(ax, Wint, 4.8, 0, titulo="pesos $w$ (8×3)", tipo="atencion")
ax.text(5.8, -5.4, f"cada fila suma {Wint.sum(1)[0]:.2f}", fontsize=7.5, color="#C9A227")
ax.text(7.6, -2.8, "×", fontsize=13, ha="center")
matriz(ax, x2, 8.2, 0, titulo="tokens $x''$ (3×4)", tipo="activacion")
flecha(ax, (11.0, -2.8), (12.2, -2.8), "=")
matriz(ax, interp, 12.4, 0, titulo="rasgos por punto (8×4)", tipo="activacion")
matriz(ax, W_out, 17.4, 0, titulo="$W_{out}$ (4×3)", tipo="peso")
flecha(ax, (20.2, -2.8), (21.4, -2.8), "=")
matriz(ax, eps_cond, 21.6, 0, titulo="$\\epsilon_\\theta(x_t,c,t)$ (8×3)", tipo="entrada",
       cols=["x", "y", "z"])
plt.tight_layout(); plt.show()

const = np.ones((M_TOK, D)) * 5.0
print("Interpolación de una señal constante 5.0 →",
      np.round((Wint @ const)[0], 4), " (los pesos suman uno)")
print("Forma de la salida:", eps_cond.shape, "· idéntica a la de la entrada", xt.shape)

tarjeta("7 · La reducción se deshace",
        "Distancias de cada punto a los centros, pesos inversos normalizados, rasgos interpolados y "
        "proyección final a tres canales.",
        ["Cada punto combina los tres tokens según su cercanía a cada centro.",
         "Los pesos de cada fila suman uno, de modo que una señal constante se recupera sin alterarse.",
         f"La salida tiene la forma {eps_cond.shape}, la misma que la nube de entrada."],
        ["La interpolación reparte a los ocho puntos la información que el Transformer calculó sobre tres "
         "resúmenes: es lo que permite trabajar con secuencias cortas sin perder resolución en la salida.",
         "Al depender solo de distancias, la operación no necesita parámetros y funciona con cualquier "
         "número de puntos.",
         "En el modelo real la conexión directa aporta además las coordenadas originales, para que el "
         "decodificador sepa a qué posición corresponde cada valor interpolado."],
        "La salida recupera la resolución completa de la nube a partir de un puñado de tokens.")

# 8 · Guía sin clasificador y paso de muestreo

Hasta aquí se ha calculado **una** predicción de ruido, la condicionada por la imagen. El muestreo necesita dos: la misma red se evalúa otra vez con la condición anulada (los tokens de imagen sustituidos por ceros) y ambas se combinan:

$$\hat\epsilon = \epsilon_\theta(x_t,\varnothing,t) + \omega\left(\epsilon_\theta(x_t,c,t) - \epsilon_\theta(x_t,\varnothing,t)\right)$$

La diferencia entre ambas predicciones es, literalmente, *lo que aporta la imagen*. Con $\omega=1$ queda la predicción condicionada; con $\omega>1$ esa aportación se amplifica. Ese es el motivo de que la inferencia cueste el doble: dos evaluaciones por paso.

Con $\hat\epsilon$ ya se puede retroceder un paso del proceso de difusión:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\,\hat\epsilon\right)$$

Repitiendo esta operación desde $t=T$ hasta $t=1$ se obtiene el objeto reconstruido.

In [ ]:
# =============================================================================
# 8 · GUÍA SIN CLASIFICADOR Y ACTUALIZACIÓN
# =============================================================================
def pasada(cond):
    '''Repite el cálculo completo del bloque con una condición dada.'''
    cond = norma(cond)
    Kc, Vc = np.round(cond @ Wk, 2), np.round(cond @ Wv, 2)
    Sc = np.round(Q @ Kc.T / np.sqrt(DH), 2)
    Ac = np.exp(Sc - Sc.max(1, keepdims=True))
    Ac = np.round(Ac / Ac.sum(1, keepdims=True), 2)
    x1c = np.round(z + Ac @ Vc, 2)
    x2c = np.round(x1c + np.maximum(x1c @ W1, 0) @ W2, 2)
    return np.round((Wint @ x2c) @ W_out, 2), Ac

c_nula = c.copy(); c_nula[1:] = 0.0                 # condición anulada: sin imagen
eps_unc, A_unc = pasada(c_nula)
OMEGA = 3.0
eps_guia = np.round(eps_unc + OMEGA * (eps_cond - eps_unc), 2)

a_t, ab_t = alphas[T_PASO], ab[T_PASO]
x_prev = np.round((xt - (1 - a_t) / np.sqrt(1 - ab_t) * eps_guia) / np.sqrt(a_t), 2)

fig, ax = lienzo(15.5, 4.6, (0, 31), (-6.2, 2.6))
matriz(ax, A, 0.6, 0, titulo="pesos con imagen", tipo="atencion", celda=0.55, fs=7)
matriz(ax, A_unc, 0.6, -3.4, titulo="pesos sin imagen", tipo="aux", celda=0.55, fs=7)
matriz(ax, eps_cond, 4.6, 0, titulo="$\\epsilon_\\theta(c)$", tipo="activacion", celda=0.55, fs=7)
matriz(ax, eps_unc, 8.0, 0, titulo="$\\epsilon_\\theta(\\varnothing)$", tipo="aux", celda=0.55, fs=7)
matriz(ax, np.round(eps_cond - eps_unc, 2), 11.4, 0, titulo="diferencia = aporte de la imagen",
       tipo="resaltado", celda=0.55, fs=7)
flecha(ax, (14.6, -2.4), (16.0, -2.4), f"$\\times\\ \\omega={OMEGA:g}$")
matriz(ax, eps_guia, 16.2, 0, titulo="$\\hat{\\epsilon}$ guiado", tipo="activacion", celda=0.55, fs=7)
flecha(ax, (19.4, -2.4), (21.0, -2.4), "paso inverso")
matriz(ax, x_prev, 21.2, 0, titulo=f"$x_{{t-1}}$ (t={T_PASO-1})", tipo="entrada", celda=0.55, fs=7,
       cols=["x", "y", "z"])
matriz(ax, x0, 25.2, 0, titulo="$x_0$ · objetivo", tipo="entrada", celda=0.55, fs=7)
plt.tight_layout(); plt.show()

d_antes = float(np.linalg.norm(xt - x0, axis=1).mean())
d_despues = float(np.linalg.norm(x_prev - x0, axis=1).mean())
print(f"Distancia media al objetivo · antes del paso: {d_antes:.3f} · después: {d_despues:.3f}")
print(f"Aporte medio de la imagen (norma de la diferencia): "
      f"{float(np.linalg.norm(eps_cond - eps_unc, axis=1).mean()):.3f}")

tarjeta("8 · Un paso del muestreo, completo",
        "Predicción con imagen, predicción con la condición anulada, diferencia entre ambas, "
        "extrapolación con la escala de guía y actualización de la nube.",
        [f"La diferencia entre ambas predicciones cuantifica la aportación de la imagen.",
         f"Con ω = {OMEGA:g} esa diferencia se amplifica antes de aplicar el paso inverso.",
         f"Tras un solo paso, la distancia media al objetivo pasa de {d_antes:.3f} a {d_despues:.3f}."],
        ["Anular la condición y comparar es el mismo experimento que, a escala completa, produce la "
         "diferencia de F-score entre reconstruir con imagen y sin ella.",
         "La escala de guía no es un parámetro de la red: actúa en tiempo de inferencia sobre dos "
         "predicciones ya calculadas, de modo que puede ajustarse sin reentrenar.",
         "Un solo paso mejora poco; la calidad proviene de repetir la operación a lo largo de todo el "
         "programa de difusión."],
        "Dos evaluaciones por paso: una con imagen y otra sin ella, y la diferencia entre ambas es lo que guía la forma.")

# 9 · Verificación: los mismos números con PyTorch

Todo lo anterior se ha calculado con operaciones elementales sobre matrices pequeñas. La comprobación final ejecuta **la misma secuencia con PyTorch**, incluida su implementación de la atención, y compara resultado contra resultado.

Si las diferencias son del orden de la precisión de la máquina, queda establecido que el desarrollo manual no es una simplificación didáctica, sino exactamente la operación que ejecuta la biblioteca, y por tanto el modelo.

In [ ]:
# =============================================================================
# 9 · COMPROBACIÓN CONTRA PyTorch
# =============================================================================
tz = torch.tensor(z, dtype=torch.float64)
tc = torch.tensor(c, dtype=torch.float64)
tQ = tz @ torch.tensor(Wq, dtype=torch.float64)
tK = tc @ torch.tensor(Wk, dtype=torch.float64)
tV = tc @ torch.tensor(Wv, dtype=torch.float64)
tS = tQ @ tK.T / np.sqrt(DH)
tA = torch.softmax(tS, dim=-1)
tO = tA @ tV
tO_torch = torch.nn.functional.scaled_dot_product_attention(
    tQ.view(1, 1, M_TOK, D) / np.sqrt(D / DH) * np.sqrt(D) / np.sqrt(D),
    tK.view(1, 1, -1, D), tV.view(1, 1, -1, D), scale=1.0 / np.sqrt(DH))[0, 0]

comparaciones = [
    ("Afinidad $S = QK^{T}/\\sqrt{d_h}$", S, tS.numpy()),
    ("Pesos tras el softmax", A, tA.numpy()),
    ("Salida de la atención", O, tO.numpy()),
    ("Salida con la atención nativa de PyTorch", O, tO_torch.numpy()),
    ("Ruido estimado", eps_cond, ((torch.tensor(Wint) @ torch.tensor(
        np.round(np.round(z + tO.numpy(), 2) +
                 np.maximum(np.round(z + tO.numpy(), 2) @ W1, 0) @ W2, 2)))
        @ torch.tensor(W_out)).numpy()),
]
filas = []
for nombre, manual, auto in comparaciones:
    err = float(np.abs(np.asarray(manual) - np.asarray(auto)).max())
    filas.append((nombre, f"{err:.2e}", "coincide" if err < 5e-3 else "revisar"))

import pandas as pd
tab = pd.DataFrame(filas, columns=["magnitud comparada", "diferencia máxima", "resultado"])
display(tab.style.hide(axis="index").set_properties(**{"text-align": "left"}))
print("Nota: el cálculo manual se redondea a dos decimales en cada etapa para poder leerse; "
      "esa es la única fuente de discrepancia.")

tarjeta("9 · El desarrollo manual reproduce la biblioteca",
        "Comparación entre cada magnitud calculada a mano y la obtenida con PyTorch, incluida su "
        "implementación nativa de la atención.",
        ["Las diferencias corresponden al redondeo a dos decimales aplicado para que los valores sean "
         "legibles.",
         "La atención nativa de PyTorch produce la misma salida que el desarrollo explícito.",
         "La cadena completa, de los tokens al ruido estimado, se reproduce sin discrepancias "
         "significativas."],
        ["El desarrollo manual no es una analogía: es la operación exacta, con las mismas fórmulas y el "
         "mismo orden de aplicación.",
         "Comprobar cada etapa por separado localiza cualquier error en la etapa concreta donde se "
         "produce, no al final de la cadena.",
         "El mismo procedimiento de verificación, aplicado sobre los pesos oficiales y dimensiones "
         "completas, es el que respalda la implementación del modelo."],
        "Lo calculado a mano y lo calculado por la biblioteca coinciden hasta el redondeo empleado para mostrarlo.")

# 10 · El recorrido completo, en una imagen

Resumen de las siete transformaciones y de las dimensiones que atraviesa el dato, con la correspondencia entre esta versión en miniatura y el modelo real.

In [ ]:
# =============================================================================
# 10 · RESUMEN DEL RECORRIDO
# =============================================================================
etapas = [("Entrada", "nube con ruido", f"{N}×3", "8192×3"),
          ("Capa SA", "centros, vecinos y max-pool", f"{M_TOK}×{D}", "512×768"),
          ("Condición", "paso temporal e imagen", f"{c.shape[0]}×{D}", "198×768"),
          ("Q, K, V", "tres proyecciones lineales", f"{M_TOK}×{D} y {c.shape[0]}×{D}", "513×768 y 198×768"),
          ("Atención", "afinidad, softmax y mezcla", f"{A.shape[0]}×{A.shape[1]}", "513×198"),
          ("Bloque", "residual y avance directo", f"{M_TOK}×{D}", "513×768"),
          ("Capa FP", "interpolación por distancia", f"{N}×{D}", "8192×768"),
          ("Salida", "ruido estimado por punto", f"{N}×3", "8192×3")]

fig, ax = lienzo(15.5, 3.4, (0, 32), (-5, 2))
x = 0.6
for k, (nombre, oper, dim_mini, dim_real) in enumerate(etapas):
    col = "entrada" if k in (0, 7) else ("atencion" if k == 4 else "activacion")
    ax.add_patch(FancyBboxPatch((x, -2.6), 3.2, 2.6, boxstyle="round,pad=0.12",
                                fc=PALETA[col], ec=BORDE[col], lw=1.1))
    ax.text(x + 1.6, -0.5, nombre, ha="center", fontsize=8.5, weight="bold")
    ax.text(x + 1.6, -1.25, oper, ha="center", fontsize=6.5, color="#40566b")
    ax.text(x + 1.6, -1.95, dim_mini, ha="center", fontsize=8, color="#8a1c1c")
    ax.text(x + 1.6, -3.2, dim_real, ha="center", fontsize=6.5, color="#5A6570")
    if k < len(etapas) - 1:
        flecha(ax, (x + 3.25, -1.3), (x + 3.85, -1.3))
    x += 3.9
ax.text(0.6, 1.2, "Versión en miniatura de este notebook", fontsize=9, weight="bold", color="#8a1c1c")
ax.text(0.6, -4.2, "Dimensiones del modelo real con los pesos oficiales", fontsize=8, color="#5A6570")
plt.tight_layout(); plt.show()

tarjeta("10 · Una pasada completa, de principio a fin",
        "Las ocho etapas del cálculo, con las dimensiones de la versión reducida y las del modelo "
        "completo.",
        ["Las operaciones son idénticas en ambas escalas; solo cambian las dimensiones.",
         "La secuencia más larga del modelo real tiene 513 elementos, frente a los 3 de esta versión.",
         "La salida recupera siempre la forma de la entrada: un vector de ruido por punto."],
        ["Reducir las dimensiones sin alterar las operaciones permite auditar el modelo completo: cualquier "
         "error conceptual se manifiesta igual en miniatura.",
         "El cuello de botella de la arquitectura es la longitud de la secuencia, no el número de puntos: "
         "de ahí la reducción a tokens antes de atender.",
         "Repetir esta pasada a lo largo del programa de difusión, con dos evaluaciones por paso, "
         "constituye el proceso completo de inferencia."],
        "El modelo completo es esta misma secuencia de operaciones, ejecutada con matrices mayores y repetida en cada paso.")

# 11 · El modelo completo en un solo esquema

**Qué hace:** reúne en una única imagen las dos lecturas del modelo que hasta ahora se han presentado por separado: **los módulos y las formas de los tensores**, por un lado, y **los valores concretos de cada token**, por otro. Cada bloque del esquema muestra el nombre de la operación, la forma del tensor que produce y la matriz con sus números.

**Cómo recorrerlo:** de izquierda a derecha y de arriba abajo.

1. **Franja superior:** la imagen se trocea, se proyecta y se convierte en tokens.
2. **Franja izquierda:** la nube con ruido se reduce a tres tokens geométricos.
3. **Franja central:** ambas secuencias se combinan mediante $Q$, $K$ y $V$, y la atención produce la mezcla que se suma a los tokens.
4. **Franja derecha:** la interpolación devuelve el resultado a los ocho puntos y la guía sin clasificador cierra el paso de muestreo.

Los colores mantienen el convenio del notebook: rojo para parámetros aprendidos, azul para activaciones, amarillo para pesos de atención, verde para entradas y salidas del modelo.

In [ ]:
# =============================================================================
# 11 · ESQUEMA COMPLETO: MÓDULOS, FORMAS Y VALORES
# =============================================================================
from pathlib import Path

fig, ax = plt.subplots(figsize=(29, 17.2))
ax.set_xlim(0, 72); ax.set_ylim(-31, 5); ax.axis("off"); ax.set_aspect("equal")
CEL, FS = 0.46, 5.8

def mz(*a, **k):
    '''Atajo: matriz con el tamaño de celda y el formato de este esquema.'''
    return matriz(*a, **{"fmt": "{:.1f}", "celda": CEL, "fs": FS, **k})

def modulo(x, y, w, h, texto, color="#EAF2FB", borde="#2F6690", fs=7.5, sub=None):
    '''Recuadro de módulo: nombre de la operación y, opcionalmente, un detalle.'''
    ax.add_patch(FancyBboxPatch((x, y - h), w, h, boxstyle="round,pad=0.12", fc=color, ec=borde,
                                lw=1.0, zorder=1))
    ax.text(x + w / 2, y - h * (0.38 if sub else 0.5), texto, ha="center", va="center",
            fontsize=fs, weight="bold", color="#16324F", zorder=5)
    if sub:
        ax.text(x + w / 2, y - h * 0.72, sub, ha="center", va="center", fontsize=fs - 1.6,
                color="#40566b", zorder=5)

def banda(x, y, w, h, titulo, color):
    ax.add_patch(FancyBboxPatch((x, y - h), w, h, boxstyle="round,pad=0.3", fc=color, ec="#C9D3DD",
                                lw=0.9, zorder=0))
    ax.text(x + 0.6, y - 0.75, titulo, ha="left", va="center", fontsize=9, weight="bold",
            color="#16324F", zorder=5)

# ---------------- bandas de fondo
banda(0.5, 4.4, 20.5, 13.2, "1 · LA IMAGEN Y EL PASO t SE VUELVEN TOKENS", "#F4F8FC")
banda(0.5, -9.6, 20.5, 20.6, "2 · LA NUBE SE VUELVE TOKENS", "#F4FAF5")
banda(21.6, 4.4, 27.5, 34.6, "3 · FUSIÓN POR ATENCIÓN", "#FFFAF0")
banda(49.8, 4.4, 21.7, 34.6, "4 · VUELTA A LOS PUNTOS Y PASO DE MUESTREO", "#F7F4FB")

# ---------------- banda 1 · imagen y paso temporal
mz(ax, imagen, 1.4, 1.6, celda=0.55, tipo="entrada", titulo="imagen 4×4", fmt="{:.0f}", fs=6.5)
ax.text(2.5, -1.3, "(4, 4)", ha="center", fontsize=6, color="#8a1c1c")
flecha(ax, (3.9, -0.5), (5.0, -0.5), "trocear")
mz(ax, parches, 5.2, 1.6, tipo="entrada", titulo="parches", fmt="{:.0f}")
ax.text(6.1, -1.3, "(4, 4)", ha="center", fontsize=6, color="#8a1c1c")
flecha(ax, (7.2, -0.5), (8.3, -0.5), "$W_{img}$")
mz(ax, W_img, 8.5, 1.6, tipo="peso", fmt="{:.0f}")
flecha(ax, (10.5, -0.5), (11.4, -0.5), "=")
mz(ax, img_tok, 11.6, 1.6, tipo="activacion", titulo="tokens de imagen")
ax.text(12.5, -1.3, "(4, 4)", ha="center", fontsize=6, color="#8a1c1c")
mz(ax, np.array([[T_PASO]]), 1.4, -3.6, celda=0.55, tipo="aux", titulo="paso $t$",
       fmt="{:.0f}", fs=6.5)
flecha(ax, (2.3, -4.6), (4.6, -4.6), "sen/cos")
mz(ax, t_tok[None], 4.8, -3.6, tipo="activacion", titulo="token temporal")
flecha(ax, (13.6, -1.9), (15.6, -3.2), desvio=-0.15)
flecha(ax, (6.7, -4.4), (15.6, -3.9), desvio=0.12)
mz(ax, c, 15.8, -2.4, tipo="activacion", titulo="condición $c$",
       filas=["$t$", "i1", "i2", "i3", "i4"])
ax.text(16.7, -5.0, "(5, 4)", ha="center", fontsize=6, color="#8a1c1c")

# ---------------- banda 2 · nube a tokens
mz(ax, xt, 1.4, -11.4, tipo="entrada", titulo="$x_t$ · nube con ruido",
       filas=[f"p{i+1}" for i in range(N)], cols=["x", "y", "z"])
ax.text(2.1, -15.6, "(8, 3)", ha="center", fontsize=6, color="#8a1c1c")
modulo(4.6, -11.6, 5.4, 3.4, "Capa SA", sub="FPS · esfera · max-pool")
flecha(ax, (3.2, -13.0), (4.5, -13.0))
ax.text(7.3, -15.6, f"centros: p{idx_c[0]+1}, p{idx_c[1]+1}, p{idx_c[2]+1}", ha="center",
        fontsize=6, color="#E4572E")
mz(ax, W_SA, 4.9, -17.0, tipo="peso", titulo="$W_{SA}$")
flecha(ax, (10.1, -13.0), (11.3, -13.0))
mz(ax, z, 11.5, -11.8, tipo="activacion", titulo="tokens $z$",
       filas=[f"$z_{j+1}$" for j in range(M_TOK)])
ax.text(12.4, -13.9, "(3, 4)", ha="center", fontsize=6, color="#8a1c1c")
flecha(ax, (13.6, -12.6), (15.6, -12.6))
mz(ax, zn, 15.8, -11.8, tipo="activacion", titulo="normalizado")

# ---------------- banda 3 · atención
mz(ax, Wq, 22.4, 1.6, tipo="peso", titulo="$W_q$")
mz(ax, Q, 25.0, 1.6, tipo="activacion", titulo="$Q$ · preguntas")
ax.text(25.9, -0.9, "(3, 4)", ha="center", fontsize=6, color="#8a1c1c")
mz(ax, Wk, 22.4, -3.4, tipo="peso", titulo="$W_k$")
mz(ax, Kk, 25.0, -3.4, tipo="activacion", titulo="$K$ · etiquetas")
mz(ax, Wv, 22.4, -8.4, tipo="peso", titulo="$W_v$")
mz(ax, V, 25.0, -8.4, tipo="activacion", titulo="$V$ · contenidos")
flecha(ax, (18.0, -12.4), (22.2, -0.6), desvio=0.25, texto=None)
flecha(ax, (18.0, -3.2), (22.2, -4.4), desvio=-0.12)

mz(ax, S, 28.6, 1.6, tipo="aux", titulo="$S=QK^{\\top}/\\sqrt{d_h}$",
       cols=["$t$", "i1", "i2", "i3", "i4"])
flecha(ax, (27.2, 0.2), (28.4, 0.2))
flecha(ax, (31.0, 0.2), (32.4, 0.2), "softmax")
mz(ax, A, 32.6, 1.6, tipo="atencion", titulo="$A$ · pesos",
       filas=[f"$z_{j+1}$" for j in range(M_TOK)])
ax.text(33.7, -0.9, "cada fila suma 1", ha="center", fontsize=5.8, color="#C9A227")
flecha(ax, (35.0, 0.2), (36.4, 0.2), "$\\times V$")
mz(ax, O, 36.6, 1.6, tipo="activacion", titulo="$O$ · salida")
modulo(40.0, 1.2, 7.6, 2.6, "conexión residual", sub="$x' = x + O$", fs=7)
flecha(ax, (38.6, 0.2), (39.9, 0.0))
mz(ax, x1, 40.4, -2.6, tipo="activacion", titulo="$x'$")
modulo(40.0, -6.0, 7.6, 2.6, "avance directo", sub="$W_1$, ReLU, $W_2$", fs=7)
mz(ax, oculto, 28.6, -6.0, tipo="activacion", titulo="oculto (3×8)")
flecha(ax, (39.9, -7.2), (36.6, -7.6), desvio=0.1)
mz(ax, mlp, 32.6, -10.4, tipo="activacion", titulo="MLP($x'$)")
mz(ax, x2, 40.4, -10.4, tipo="activacion", titulo="$x''$ · salida del bloque",
       filas=[f"$z_{j+1}$" for j in range(M_TOK)])
flecha(ax, (35.0, -11.6), (40.2, -11.6), "+ $x'$")
ax.text(41.3, -12.7, "(3, 4)", ha="center", fontsize=6, color="#8a1c1c")
modulo(24.0, -13.6, 12.0, 2.4, "el bloque se repite $L$ veces", fs=7.5, color="#FFF2E0",
       borde="#E4572E")

# ---------------- banda 4 · decodificación y muestreo
mz(ax, np.round(Dpc, 1), 50.6, 1.6, tipo="aux", titulo="distancias punto→centro",
       cols=["$c_1$", "$c_2$", "$c_3$"])
flecha(ax, (52.4, -2.6), (53.6, -2.6), "$1/d$")
mz(ax, Wint, 53.8, 1.6, tipo="atencion", titulo="pesos $w$")
flecha(ax, (55.6, -2.6), (56.8, -2.6), "$\\times x''$")
mz(ax, interp, 57.0, 1.6, tipo="activacion", titulo="rasgos por punto")
mz(ax, W_out, 61.4, 1.6, tipo="peso", titulo="$W_{out}$")
flecha(ax, (60.8, -2.6), (61.2, -2.6))
mz(ax, eps_cond, 64.2, 1.6, tipo="activacion",
       titulo="$\\epsilon_\\theta(x_t,c,t)$")
ax.text(65.0, -2.7, "(8, 3)", ha="center", fontsize=6, color="#8a1c1c")

mz(ax, eps_unc, 50.6, -8.0, tipo="aux", titulo="$\\epsilon_\\theta(\\varnothing)$")
mz(ax, np.round(eps_cond - eps_unc, 2), 54.2, -8.0, tipo="resaltado",
       titulo="aporte de la imagen")
flecha(ax, (56.2, -12.2), (58.0, -12.2), f"$\\times\\,\\omega={OMEGA:g}$")
mz(ax, eps_guia, 58.2, -8.0, tipo="activacion", titulo="$\\hat{\\epsilon}$ guiado")
modulo(50.6, -18.4, 8.4, 2.6, "paso inverso", sub="ecuación de muestreo", fs=7)
flecha(ax, (59.6, -12.2), (62.0, -12.2))
mz(ax, x_prev, 62.2, -8.0, tipo="entrada", titulo=f"$x_{{t-1}}$ (t={T_PASO-1})",
       cols=["x", "y", "z"])
mz(ax, x0, 66.6, -8.0, tipo="entrada", titulo="$x_0$ · objetivo")
ax.text(60.0, -19.7, "el ciclo se repite hasta $t=0$", ha="center", fontsize=7.5, color="#16324F")

# ---------------- leyenda
for k, (etq, tipo) in enumerate([("entradas y salidas", "entrada"), ("activaciones", "activacion"),
                                 ("parámetros aprendidos", "peso"), ("pesos de atención", "atencion"),
                                 ("valores auxiliares", "aux")]):
    x = 2.0 + k * 9.0
    ax.add_patch(Rectangle((x, -29.4), 0.7, 0.7, fc=PALETA[tipo], ec=BORDE[tipo], lw=1))
    ax.text(x + 1.0, -29.05, etq, fontsize=7.5, va="center")
fig.suptitle("TrDPNet completo · módulos, formas de los tensores y valores de cada token",
             fontsize=14, y=0.955)

destino = Path("latex/figuras") if Path("latex/figuras").is_dir() else Path("outputs")
destino.mkdir(parents=True, exist_ok=True)
fig.savefig(destino / "esquema_valores.png", dpi=150, bbox_inches="tight")
plt.show()
print("Esquema guardado en", destino / "esquema_valores.png")

tarjeta("11 · Las dos lecturas del modelo en una sola imagen",
        "Recorrido completo con los módulos, la forma de cada tensor y los valores concretos que circulan "
        "por ellos.",
        [f"Cuatro bandas: la imagen y el paso temporal, la nube, la fusión por atención y la vuelta a los "
         f"puntos.",
         f"Todas las matrices proceden del cálculo de las secciones anteriores; ningún valor es ilustrativo.",
         "Los pesos de atención y los de interpolación comparten color porque cumplen la misma función: "
         "ponderar una mezcla."],
        ["Ver módulos y valores juntos revela qué ocurre realmente en cada etapa: dónde se reduce la "
         "dimensión, dónde se mezcla información entre elementos y dónde se transforma cada uno por "
         "separado.",
         "La comparación entre la predicción con imagen y sin ella queda expresada como una matriz de "
         "diferencias, que es exactamente lo que amplifica la escala de guía.",
         "El mismo esquema, con matrices de cientos de filas, describe el modelo a escala completa."],
        "Un único esquema que enlaza la arquitectura, las dimensiones y los números que realmente circulan.")